# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['HZSEVZJAOV', 'YFYVQUJZRG'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8, 26, 19,  5, 22, 26, 10,  1, 15, 22],
       [25,  6, 25, 22, 17, 21, 10, 26, 18,  7]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 22, 15,  1, 10, 26, 22,  5, 19, 26],
       [ 0,  7, 18, 26, 10, 21, 17, 22, 25,  6]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[22, 15,  1, 10, 26, 22,  5, 19, 26,  8],
       [ 7, 18, 26, 10, 21, 17, 22, 25,  6, 25]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [12]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 256
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 128, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense_context = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # Encoder: 获取 encoder 输出和状态
        enc_out, [enc_last, enc_state] = self.encode(enc_ids)  # enc_out: (b_sz, seq_len, hidden)
        
        # Decoder embedding
        dec_emb = self.embed_layer(dec_ids)  # (b_sz, dec_len, emb_sz)
        
        # 双线性 Attention: scores = h_dec^T * W * h_enc
        # dec_proj: (b_sz, dec_len, hidden)
        dec_proj = self.dense_attn(dec_emb)
        # scores: (b_sz, dec_len, enc_len)
        scores = tf.matmul(dec_proj, enc_out, transpose_b=True)
        
        # 计算 attention 权重
        attn_weights = tf.nn.softmax(scores, axis=-1)  # (b_sz, dec_len, enc_len)
        
        # 计算 context vector: 加权求和
        context = tf.matmul(attn_weights, enc_out)  # (b_sz, dec_len, hidden)
        
        # 将 context 与 decoder embedding 拼接
        dec_input = tf.concat([dec_emb, context], axis=-1)  # (b_sz, dec_len, emb_sz + hidden)
        
        # 通过 dense 将维度映射回 hidden
        dec_input = self.dense_context(dec_input)  # (b_sz, dec_len, hidden)
        
        # Decoder RNN
        dec_out, dec_state = self.decoder(dec_input)  # dec_out: (b_sz, dec_len, hidden)
        
        # 输出 logits
        logits = self.dense(dec_out)  # (b_sz, dec_len, v_sz)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
        # Embedding
        inp_emb = self.embed_layer(x)  # shape(b_sz, emb_sz)
        inp_emb_exp = tf.expand_dims(inp_emb, axis=1)  # shape(b_sz, 1, emb_sz)
        
        # 双线性 Attention
        dec_proj = self.dense_attn(inp_emb_exp)  # (b_sz, 1, hidden)
        # scores: (b_sz, 1, enc_len)
        scores = tf.matmul(dec_proj, enc_out, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)  # (b_sz, 1, enc_len)
        
        # 计算 context vector
        context = tf.matmul(attn_weights, enc_out)  # (b_sz, 1, hidden)
        
        # 拼接 embedding 和 context
        inp = tf.concat([inp_emb_exp, context], axis=-1)  # (b_sz, 1, emb_sz + hidden)
        inp = self.dense_context(inp)  # (b_sz, 1, hidden)
        
        # Decoder cell 单步计算
        h, new_state = self.decoder_cell.call(inp[:, 0, :], state)  # shape(b_sz, hidden)
        
        # 输出 logits
        logits = self.dense(h)  # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, new_state

# Loss函数以及训练逻辑

In [20]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(20000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [21]:
optimizer = optimizers.Adam(0.001)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.30358
step 500 : loss 0.31864676
step 1000 : loss 0.17460783
step 1500 : loss 0.030121922
step 2000 : loss 0.014021722
step 2500 : loss 0.017810453
step 3000 : loss 0.041044973
step 3500 : loss 0.019434
step 4000 : loss 0.010515005
step 4500 : loss 0.012315961
step 5000 : loss 0.014620108
step 5500 : loss 0.024136718
step 6000 : loss 0.0278676
step 6500 : loss 0.024479512
step 7000 : loss 0.019350065
step 7500 : loss 0.03266428
step 8000 : loss 0.034649037
step 8500 : loss 0.011032891
step 9000 : loss 0.005430766
step 9500 : loss 0.014632088
step 10000 : loss 0.01862835
step 10500 : loss 0.0063375025
step 11000 : loss 0.011133306
step 11500 : loss 0.0034765645
step 12000 : loss 0.0061009796
step 12500 : loss 0.008453404
step 13000 : loss 0.021353185
step 13500 : loss 0.0041082716
step 14000 : loss 0.009948501
step 14500 : loss 0.0051490464
step 15000 : loss 0.0033765503
step 15500 : loss 0.087968916
step 16000 : loss 0.01389471
step 16500 : loss 0.0053806035
step 17000 

<tf.Tensor: shape=(), dtype=float32, numpy=0.006140046>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [22]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True]
[('EAFTBNYZCUFFLEYIPSGT', 'TGSPIYELFFUCZYNBTFAE'), ('TQTVKIXDCGVJSEDTAFAT', 'TAFATDESJVGCDXIKVTQT'), ('DHNNQCBSCWKLOSHURLGK', 'KGLRUHSOLKWCSBCQNNHD'), ('PZYEEGLRIMIHWKKBVJDU', 'UDJVBKKWHIMIRLGEEYZP'), ('HBHNVYFVPGBHGVFNRLCK', 'KCLRNFVGHBGPVFYVNHBH'), ('OOFVFDAFLILRHSPHJVJR', 'RJVJHPSHRLILFADFVFOO'), ('KIOYXSBBPRJRCJXSSLWM', 'MWLSSXJCRJRPBBSXYOIK'), ('DGVSEOFSNJRYWDPNRNCY', 'YCNRNPDWYRJNSFOESVGD'), ('SLMZLLUQDYRWWSARMVCF', 'FCVMRASWWRYDQULLZMLS'), ('UWPOBQENCGNNDCYSUFOE', 'EOFUSYCDNNGCNEQBOPWU'), ('VCMFPWIMPGWZLJJWNXBL', 'LBXNWJJLZWGPMIWPFMCV'), ('BIAOIZULEFGZFIEVLFMC', 'CMFLVEIFZGFELUZIOAIB'), ('ASWCNLLMRVFXDMUIUGEA', 'AEGUIUMDXFVRMLLNCWSA'), ('WLJVVIHSYKZRVUIUQUVB', 'BVUQUIUVRZKYSHIVVJLW'), ('NLUGDNYVUEJIUZAUPGOB', 'BOGPUAZUIJEUVYNDGULN'), ('HOTMSVFNSLWUJAWKVXSI', 'ISXVKWAJUWLSNFVSMTOH'), ('XM